# Baseline

### 필요 라이브러리 설치

In [4]:
%pip install pandas numpy scikit-learn lightgbm

Note: you may need to restart the kernel to use updated packages.


### LightGBM 모델 학습 및 자체 평가

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. 데이터 불러오기
print("데이터 로딩 중... (용량이 커서 몇 초 정도 걸릴 수 있습니다)")
train = pd.read_csv('./data/train.csv')

# 2. 결측치 처리 (이전에 계획한 단순 대체 방법)
print("결측치 처리 중...")
# 숫자형 결측치는 평균으로, 범주형 결측치는 'Unknown'으로 채웁니다.
numeric_cols = train.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if train[col].isnull().any():
        train[col] = train[col].fillna(train[col].mean())

object_cols = train.select_dtypes(include=['object']).columns
for col in object_cols:
    if train[col].isnull().any():
        train[col] = train[col].fillna('Unknown')

# 3. Train / Validation 데이터 나누기 (매우 중요!)
print("학습용/검증용 데이터 분리 중...")
# 대회 평가에 제출할 'row_id'는 예측에 방해되므로 제외하고, 정답지인 'control_success'도 분리합니다.
X = train.drop(columns=['control_success', 'row_id'])
y = train['control_success']

# 모델 학습용 80%, 로컬 검증용 20%로 나눕니다.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42 # 항상 동일하게 나뉘도록 시드 고정
)

print(f"✅ 완료! 학습용 데이터 크기: {X_train.shape}, 검증용 데이터 크기: {X_valid.shape}")

데이터 로딩 중... (용량이 커서 몇 초 정도 걸릴 수 있습니다)
결측치 처리 중...
학습용/검증용 데이터 분리 중...


/var/folders/hm/v0t_41rn1k745bstq7qk0tn40000gn/T/ipykernel_93659/3416133666.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = train.select_dtypes(include=['object']).columns


✅ 완료! 학습용 데이터 크기: (1180073, 47), 검증용 데이터 크기: (295019, 47)


In [6]:
import lightgbm as lgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
import numpy as np
import joblib
import os

os.makedirs('./model', exist_ok=True)

print("1. 투수별 과거 제구 성공률(Target Encoding) 피처 생성 중...")
# Train 데이터 정답(y_train)을 활용해 투수별 평균 제구율을 구합니다. (가장 강력한 힌트!)
temp_train = X_train.copy()
temp_train['target'] = y_train
pitcher_stat = temp_train.groupby('pitcher_id')['target'].mean().reset_index()
pitcher_stat.rename(columns={'target': 'pitcher_control_rate'}, inplace=True)

overall_mean = y_train.mean() # 신인 투수를 위한 전체 평균값

# Train/Valid 데이터에 맵핑 및 결측치 처리
X_train = pd.merge(X_train, pitcher_stat, on='pitcher_id', how='left')
X_valid = pd.merge(X_valid, pitcher_stat, on='pitcher_id', how='left')
X_train['pitcher_control_rate'] = X_train['pitcher_control_rate'].fillna(overall_mean)
X_valid['pitcher_control_rate'] = X_valid['pitcher_control_rate'].fillna(overall_mean)

# 💡 서버에서도 똑같이 쓰기 위해 딕셔너리(사전)로 저장해 둡니다.
joblib.dump(pitcher_stat, './model/pitcher_stat.pkl')
joblib.dump(overall_mean, './model/overall_mean.pkl')

print("2. 범주형 데이터 변환 중...")
object_cols = X_train.select_dtypes(include=['object']).columns
for col in object_cols:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')

print("3. LightGBM + 확률 보정(Calibration) 모델 학습 시작...")
# 기본 LightGBM 모델
base_model = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 💡 Brier Score 핵심 치트키: CalibratedClassifierCV
# 극단적인 확률(0.99, 0.01)을 내뱉지 않도록 현실적인 확률로 깎아줍니다(보정).
model = CalibratedClassifierCV(base_model, method='isotonic', cv=3)
model.fit(X_train, y_train)

# 학습된 최종 모델 저장
joblib.dump(model, './model/model.pkl')

print("\n4. 모의고사(검증 데이터) 예측 및 채점 중...")
y_pred_proba = model.predict_proba(X_valid)[:, 1]
brier = brier_score_loss(y_valid, y_pred_proba)
bss = max(0, 100000 * (1 - brier / (overall_mean * (1 - overall_mean))))

print("="*40)
print("🎯 V2 학습 및 자체 평가 완료!")
print(f"▶ 순수 Brier Score: {brier:.4f}")
print(f"🏆 예상 환산 점수: {bss:.2f}점")
print("="*40)

1. 투수별 과거 제구 성공률(Target Encoding) 피처 생성 중...
2. 범주형 데이터 변환 중...
3. LightGBM + 확률 보정(Calibration) 모델 학습 시작...


/var/folders/hm/v0t_41rn1k745bstq7qk0tn40000gn/T/ipykernel_93659/1803502383.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = X_train.select_dtypes(include=['object']).columns


[LightGBM] [Info] Number of positive: 412201, number of negative: 374514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019816 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6645
[LightGBM] [Info] Number of data points in the train set: 786715, number of used features: 48
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.523952 -> initscore=0.095882
[LightGBM] [Info] Start training from score 0.095882
[LightGBM] [Info] Number of positive: 412201, number of negative: 374514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015747 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6642
[LightGBM] [Info] Number of data points in the train set: 786715, number of used features: 48
[LightGBM] [In

### 학습 모델 저장

In [7]:
import joblib
import os

# model 폴더가 없다면 생성합니다.
os.makedirs('./model', exist_ok=True)

# 아까 학습시킨 LightGBM 모델을 'model.pkl'이라는 이름으로 저장합니다.
joblib.dump(model, './model/model.pkl')
print("모델이 ./model/model.pkl 경로에 성공적으로 저장되었습니다")

모델이 ./model/model.pkl 경로에 성공적으로 저장되었습니다


In [8]:
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("lightgbm\n")
print("requirements.txt 파일이 성공적으로 생성되었습니다")

requirements.txt 파일이 성공적으로 생성되었습니다


### 제출용 script.py 생성

In [9]:
script_content = """import pandas as pd
import numpy as np
import joblib
import os

def main():
    print("1. 평가 데이터 불러오기...")
    test = pd.read_csv('./data/test.csv')
    
    row_ids = test['row_id']
    X_test = test.drop(columns=['row_id'])
    
    print("2. 데이터 전처리 중 (Train과 완벽히 동일하게!)...")
    # 💡결측치 불일치 버그 해결: Test 데이터도 단순 -1로 채워 모델의 혼동을 막습니다. 
    # (실무에서는 LightGBM이 -1을 결측치 그룹으로 잘 묶어서 판단합니다)
    numeric_cols = X_test.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if X_test[col].isnull().any():
            X_test[col] = X_test[col].fillna(-1)
            
    # 💡투수별 과거 제구 성공률(Target Encoding) 적용
    pitcher_stat = joblib.load('./model/pitcher_stat.pkl')
    overall_mean = joblib.load('./model/overall_mean.pkl')
    
    X_test = pd.merge(X_test, pitcher_stat, on='pitcher_id', how='left')
    X_test['pitcher_control_rate'] = X_test['pitcher_control_rate'].fillna(overall_mean)
            
    # 범주형(Object) 결측치 및 타입 변환
    object_cols = X_test.select_dtypes(include=['object']).columns
    for col in object_cols:
        if X_test[col].isnull().any():
            X_test[col] = X_test[col].fillna('Unknown')
        X_test[col] = X_test[col].astype('category')
        
    print("3. 보정된 모델(Calibrated) 불러오기...")
    model = joblib.load('./model/model.pkl')
    
    print("4. 제구 성공 확률 추론 중...")
    pred_proba = model.predict_proba(X_test)[:, 1]
    
    print("5. 결과 저장 중...")
    os.makedirs('./output', exist_ok=True)
    pd.DataFrame({
        'row_id': row_ids,
        'control_success': pred_proba
    }).to_csv('./output/submission.csv', index=False)
    print("✅ 추론 파이프라인 무사 통과!")

if __name__ == '__main__':
    main()
"""

with open("script.py", "w", encoding="utf-8") as f:
    f.write(script_content)
print("✅ V2 제출용 script.py 파일 생성 완료!")

✅ V2 제출용 script.py 파일 생성 완료!
